# 7Learnings Data Scientist Case Study: Retail Demand Forecasting

Forecast observed unit sales before returns for market `DE`, channel `myshop.de`, using the `(product_id, date)` pairs in `sample_prediction.csv` for **September 7–20, 2026**. The training cutoff is **September 6, 2026, 23:59:59**. See the README for the full task and dataset details.


## Deliverables

- Submit this notebook or a reproducible Python script/repository.
- Generate `predictions.csv` in the project directory with columns `product_id,date,predicted_sales`, covering the exact pairs in `sample_prediction.csv`.
- Include a baseline comparison, your investigation SQL and results, a compact supporting table or chart, and the client memo. Add code cells wherever you need them.


## Environment Setup & Data Preparation

Follow the README to install the dependencies and acquire the DuckDB database. Use the supplied tables and keep realized observations within the historical cutoff. The README defines which planned promotions are available for this forecast and how to handle them during historical validation.


## AI Usage & Transparency

AI tools are permitted. Disclose which tools you used and how they helped. Be prepared to explain your analysis and assumptions.

### Candidate AI Disclosure
- AI assistants used: Claude
- Tasks they assisted with: writing DuckDB SQL and pandas code, structuring the notebook, and drafting explanations and the client memo.
- Prompts or queries used: requests to validate some tables and remove anomalies, to write the code that calculates the requested featrues, to compare LightGBM against the baseline, and to test whether two week sales drops persist historically.
- Confirmation: I reviewed my submission and can explain its findings, assumptions, and decisions.


## Part 1: Data Analysis

Analyze the data’s structure, coverage, and relationships relevant to the forecast. Document any data preparation, transformations, or enrichment you choose and explain why.

### Findings and rationale

Summary in 1.5.


### 1.0 Setup

In [1]:
import duckdb
import pandas as pd

con = duckdb.connect("7ldata.duckdb", read_only=True)
sql = lambda q: con.sql(q).df()

CUTOFF = "2026-09-06 23:59:59"

### 1.1 Basic checks
The SQL equivalent of `shape`, `isna().sum()` and `duplicated().sum()` for every table, plus date range and number of products.

In [2]:
date_col = {"transactions": "time", "stock": "active_since", "traffic": "active_since",
            "promotions": "active_since", "product_attributes": None}
checks = []
for t, d in date_col.items():
    s = sql(f"SUMMARIZE {t}")
    checks.append({
        "table": t,
        "rows": sql(f"SELECT COUNT(*) FROM {t}").iat[0, 0],
        "columns": len(s),
        "duplicate_rows": sql(f"SELECT COUNT(*) - (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM {t})) FROM {t}").iat[0, 0],
        "columns_with_nulls": ", ".join(s.loc[s.null_percentage.astype(float) > 0, "column_name"]) or "-",
        "products": sql(f"SELECT COUNT(DISTINCT product_id) FROM {t}").iat[0, 0],
        "first_date": sql(f"SELECT MIN({d})::DATE FROM {t}").iat[0, 0] if d else None,
        "last_date": sql(f"SELECT MAX({d})::DATE FROM {t}").iat[0, 0] if d else None,
    })
pd.DataFrame(checks)

,table,rows,columns,duplicate_rows,columns_with_nulls,products,first_date,last_date
0,transactions,2764520,23,27370,"product_id, revenue",13385,1970-01-01,2099-12-31
1,stock,27300,6,0,"end_liquidation_value, stock_start_of_day",3900,2023-12-26,2026-09-11
2,traffic,2736950,8,0,-,3920,2024-09-16,2026-09-20
3,promotions,8340,7,0,-,3920,2026-09-01,2026-09-21
4,product_attributes,3920,23,0,"producer, color, size, gender, shelf_location",3920,NaT,NaT


**What this tells us**
- `transactions` has **27,370 exact duplicate rows**, **missing `product_id`s**, and dates of **1970 and 2099**: placeholders, not real sales. Real history starts on 2024-09-16 (two years).
- `transactions` has **13,385 distinct product ids** while the catalog has **3,920** (the same products as `sample_prediction.csv`). Section 1.2 explains the gap.
- `stock`, `traffic` and `transactions` run past the cutoff; those rows are evaluation data and are not used.
- `promotions` only covers September 2026 onwards, so there is no promotion history to learn from (Part 2).
- Other nulls (`revenue`, stock values, descriptive catalog fields such as `color` or `size`) do not affect the target and those columns are not used by the model.

### 1.2 Data quality issues in `transactions`
Each row gets the first issue that applies, so categories do not overlap.

In [3]:
sql(f"""
SELECT CASE
    WHEN time < '2024-09-16' OR time > '2026-12-31'      THEN '1 invalid date (1970 / 2099)'
    WHEN time > '{CUTOFF}'                               THEN '2 after cutoff'
    WHEN product_id IS NULL                              THEN '3 missing product_id'
    WHEN product_id IN ('TEST_SKU_001', 'INTERNAL_TEST') THEN '4 test SKU'
    WHEN sales_before_returns >= 10000                   THEN '5 placeholder quantity (10000 / 99999)'
    WHEN sales_before_returns <= 0                       THEN '6 zero or negative quantity'
    WHEN product_id NOT IN (SELECT product_id FROM product_attributes)
                                                         THEN '7 id format (case / spaces)'
    ELSE '8 ok' END AS issue,
  COUNT(*) AS rows,
  SUM(sales_before_returns) AS units,
  MAX(sales_before_returns) AS max_quantity
FROM transactions
GROUP BY 1 ORDER BY 1
""")

,issue,rows,units,max_quantity
0,1 invalid date (1970 / 2099),50,181.0,6
1,2 after cutoff,55423,409303.0,99999
2,3 missing product_id,13434,50403.0,20
3,4 test SKU,195,739.0,10
4,5 placeholder quantity (10000 / 99999),98,5299952.0,99999
5,6 zero or negative quantity,482,-2548.0,-1
6,7 id format (case / spaces),24543,93189.0,43
7,8 ok,2670295,9956734.0,42


**Decisions**
- **Drop** invalid dates, post-cutoff rows, missing ids (~0.5% of volume, cannot be assigned to a product) and test SKUs.
- **Drop quantities of 10,000 / 99,999.** The largest valid quantity is 43. These 98 rows would add **5.3M units**, inflating history by more than 50%.
- **Drop zero or negative quantities**: the target is units *before* returns.
- **Recover the id-format rows.** `2_X_669815` or `" 669107"` are catalog products written with different case or spaces; `lower(trim(product_id))` matches all of them to the catalog. This keeps 93k units that a plain join would lose.
- **Remove exact duplicates** found in 1.1 (the same order line loaded twice).

### 1.3 Stock-out period
801 products have zero stock in the 2025-11-25 and 2026-04-24 snapshots of `stock`. Their sales:

In [4]:
sql("""
WITH zero_stock AS (SELECT DISTINCT product_id FROM stock WHERE stock_start_of_day = 0)
SELECT date_trunc('month', time)::DATE AS month,
       COUNT(DISTINCT z.product_id)    AS products_selling,
       SUM(sales_before_returns)       AS units
FROM transactions t
JOIN zero_stock z ON lower(trim(t.product_id)) = z.product_id
WHERE time BETWEEN '2025-09-01' AND '2026-07-31' AND sales_before_returns BETWEEN 1 AND 1000
GROUP BY 1 ORDER BY 1
""")

,month,products_selling,units
0,2025-09-01,801,55882.0
1,2025-10-01,801,58662.0
2,2025-11-01,801,46885.0
3,2026-05-01,801,16031.0
4,2026-06-01,801,59357.0
5,2026-07-01,801,52503.0


**Decision.** December to April are missing entirely: these products stop selling on 2025-11-24 and restart on 2026-05-24, the day stock is positive again. The zeros are **missing supply, not missing demand**, so the period 2025-11-25 to 2026-05-23 is flagged as `stockout` for these 801 products and excluded from training and from historical averages. Outside this event the stock snapshots are nearly constant, so stock is not used as a feature.

### 1.4 Clean product-day table
Applies all decisions and builds the full grid of products × days, so days without sales become explicit zeros.

In [5]:
con.sql(f"""
CREATE OR REPLACE TEMP TABLE sales_clean AS
SELECT DISTINCT lower(trim(t.product_id)) AS product_id, t.time, t.order_id,
       t.basket_position, t.sales_before_returns
FROM transactions t
JOIN product_attributes p ON lower(trim(t.product_id)) = p.product_id  -- recovers id formats; drops NULL ids and test SKUs
WHERE t.time BETWEEN '2024-09-16' AND '{CUTOFF}'                        -- drops 1970/2099 and post-cutoff rows
  AND t.sales_before_returns BETWEEN 1 AND 1000                         -- drops placeholders and negatives
""")

con.sql("""
CREATE OR REPLACE TEMP TABLE daily AS
WITH grid  AS (SELECT p.product_id, r.d::DATE AS date
               FROM product_attributes p
               CROSS JOIN range(DATE '2024-09-16', DATE '2026-09-07', INTERVAL 1 DAY) r(d)),
     sales AS (SELECT product_id, time::DATE AS date, SUM(sales_before_returns) AS units
               FROM sales_clean GROUP BY 1, 2),
     zero_stock AS (SELECT DISTINCT product_id FROM stock WHERE stock_start_of_day = 0)
SELECT g.product_id, g.date,
       COALESCE(s.units, 0) AS units,
       (g.product_id IN (SELECT product_id FROM zero_stock)
        AND g.date BETWEEN '2025-11-25' AND '2026-05-23') AS stockout
FROM grid g LEFT JOIN sales s USING (product_id, date)
""")

sql("""
SELECT COUNT(*) AS rows, COUNT(DISTINCT product_id) AS products,
       MIN(date) AS first_date, MAX(date) AS last_date,
       SUM(units) AS units, SUM(stockout::INT) AS stockout_days,
       ROUND(AVG((units = 0 AND NOT stockout)::INT), 3) AS share_zero_days_in_stock
FROM daily
""")

,rows,products,first_date,last_date,units,stockout_days,share_zero_days_in_stock
0,2826320,3920,2024-09-16,2026-09-06,9949684.0,144180.0,0.005


### 1.5 Summary

**Result.** `daily` has one row per product and day (3,920 × 721 = 2.83M rows, 9.95M units). Only **0.5%** of in-stock days have zero sales: demand is dense, not intermittent.

| Issue | Decision |
|---|---|
| Invalid dates, post-cutoff rows | Drop |
| Missing `product_id`, test SKUs | Drop |
| Quantities 10,000 / 99,999 and ≤ 0 | Drop |
| Id case / spaces | Recover with `lower(trim())` |
| Exact duplicates | Remove |
| Stock-out Nov-25 to May-26 (801 products) | Flag and exclude from training |

**Assumptions and excluded data**
- Timestamps are local (German) time; products stay in stock during 7–20 September.
- `traffic` and `stock` are realised data, unknown for future days, so they are not used as future features.
- Catalog `avg_historic_sales` (unknown computation window) and current prices (today's snapshot, not historical) are not used as features, to avoid leakage.

## Part 2: Feature Engineering

Describe the features you choose from the supplied tables and how they support the forecast. See the README for the dataset schema and DuckDB tip.

### Features and rationale

Summary in 2.5.


### 2.1 Forecasting setup: what is known at the forecast origin
Every forecast is made at an **origin** date (the last day of known data) for the next 14 days.
A feature is valid only if it is known at the origin:

| Information | Known at origin? | Use |
|---|---|---|
| Past sales (up to the origin) | Yes | Recent averages, same weekday, last year |
| Calendar of the target day | Yes | Day of week, days ahead (horizon) |
| Catalog attributes | Yes (static) | Tested, did not help (see 2.5) |
| Promotion plan | Only in the final forecast | See 2.3 |
| Traffic, stock | No (realised data) | Not used, see 2.2 |

### 2.2 Signals in the data

In [6]:
import numpy as np
import warnings
warnings.filterwarnings("ignore", message="Mean of empty slice")  # all-missing windows (stock-outs) give NaN on purpose

daily = sql("SELECT * FROM daily ORDER BY product_id, date")
daily["date"] = pd.to_datetime(daily["date"])
in_stock = daily[~daily.stockout]

# Average units per product by weekday (0 = Monday)
in_stock.groupby(in_stock.date.dt.dayofweek).units.mean().round(2).rename("units_per_product")

date
0    3.83
1    3.41
2    3.08
3    3.46
4    3.79
5    4.18
6    4.23
Name: units_per_product, dtype: float64

**Day of week matters:** Wednesday is the weakest day and the weekend the strongest (about 35% more than Wednesday). The target weekday is a feature, and so is the average of the same weekday in recent weeks.

In [7]:
sql("""
WITH s AS (SELECT date, SUM(units) AS units FROM daily GROUP BY 1),
     t AS (SELECT active_since::DATE AS date, SUM(total_clicks) AS clicks FROM traffic GROUP BY 1)
SELECT ROUND(corr(units, clicks), 3) AS corr_sales_vs_clicks,
       ROUND(AVG(clicks / units), 1) AS clicks_per_unit_sold
FROM s JOIN t USING (date)
""")

,corr_sales_vs_clicks,clicks_per_unit_sold
0,0.998,253.0


**Traffic is excluded.** Daily clicks move almost one-to-one with daily sales (correlation 0.998, about 253 clicks per unit sold every day). It is a mirror of sales rather than an independent driver: its past values add nothing beyond past sales, and its future values are unknown at the origin.

### 2.3 Promotion plan

In [8]:
sql("""
SELECT promotion_name, active_since::DATE AS start, active_till::DATE AS end,
       COUNT(DISTINCT product_id) AS products, voucher_rate
FROM promotions
GROUP BY ALL ORDER BY start, promotion_name
""")

,promotion_name,start,end,products,voucher_rate
0,Voucher Campaign 10%,2026-09-01,2026-09-20,3920,0.10
1,Voucher Campaign 5%,2026-09-11,2026-09-16,3920,0.05
2,Voucher Campaign 10%,2026-09-21,2026-10-05,242,0.10
3,Voucher Campaign 5%,2026-09-21,2026-10-05,258,0.05


**No promotion feature.** The 10% voucher covers **all 3,920 products for the whole horizon** and was already running from 1 September. With no product or day variation and no earlier campaigns, a model cannot learn its effect. Instead:
- **Final forecast:** the last 6 days of history (1–6 September) were already under the campaign, so the recent-average features reflect it.
- **Assumption:** the 5% voucher (11–16 September) does not stack with the 10% one, so it adds no extra effect.
- **Validation:** no campaign was running at the validation origin (23 August), so the validation forecast uses no promotion information, as the README requires.

### 2.4 Feature builder
Sales are pivoted into a products × days matrix (stock-out days set to missing, so they are ignored by every average). For an origin and each horizon `h = 1..14`, every feature is computed **only from columns up to the origin**, so nothing inside the forecast window leaks in. One function serves training (many past origins), validation (origin 23 August) and the final forecast (origin 6 September).

In [9]:
daily.loc[daily.stockout, "units"] = np.nan
M = daily.pivot(index="product_id", columns="date", values="units")
X, dates, products = M.to_numpy(), M.columns, M.index

def build_features(origin, horizon=14):
    t = dates.get_loc(pd.Timestamp(origin))       # column of the origin day
    last = lambda k: X[:, t - k + 1 : t + 1]      # last k days up to the origin
    rows = []
    for h in range(1, horizon + 1):
        target = t + h
        first_week = -(-h // 7)                   # first weekly lag that is not after the origin
        same_dow = [target - 7 * k for k in range(first_week, first_week + 4)]
        rows.append(pd.DataFrame({
            "product_id": products,
            "date": dates[0] + pd.Timedelta(days=target),
            "horizon": h,
            "dow": (dates[0] + pd.Timedelta(days=target)).dayofweek,
            "mean_7": np.nanmean(last(7), axis=1),
            "mean_28": np.nanmean(last(28), axis=1),
            "mean_91": np.nanmean(last(91), axis=1),
            "same_dow_mean_4w": np.nanmean(X[:, same_dow], axis=1),
            "last_year": X[:, target - 364] if target >= 364 else np.nan,  # NaN before data starts
            "y": X[:, target] if target < len(dates) else np.nan,   # actual sales (target)
        }))
    return pd.concat(rows, ignore_index=True)

FEATURES = ["horizon", "dow", "mean_7", "mean_28", "mean_91", "same_dow_mean_4w", "last_year"]
build_features("2026-08-23").head()

,product_id,date,horizon,dow,mean_7,mean_28,mean_91,same_dow_mean_4w,last_year,y
0,2_x_669800,2026-08-24,1,0,5.142857,5.285714,4.219780,5.75,3.0,5.0
1,2_x_669801,2026-08-24,1,0,4.000000,4.214286,6.483516,4.75,6.0,4.0
2,2_x_669803,2026-08-24,1,0,3.285714,3.321429,3.186813,3.75,2.0,3.0
3,2_x_669804,2026-08-24,1,0,2.428571,2.785714,2.681319,3.00,2.0,2.0
4,2_x_669806,2026-08-24,1,0,3.428571,3.607143,5.670330,4.00,8.0,3.0


### 2.5 Features and rationale

| Feature | What it captures |
|---|---|
| `mean_7`, `mean_28`, `mean_91` | Product level over short, medium and long windows (recent changes vs stable level) |
| `same_dow_mean_4w` | Product sales on the target weekday over the last four weeks known at the origin |
| `dow`, `horizon` | Weekday pattern and how far ahead the forecast is |
| `last_year` | Same weekday 52 weeks earlier (`target − 364` days), always before the origin |

**Tested and dropped** (development tests on the validation windows):
- Shop-wide averages of the last 7 and 28 days: with one value per origin they take only 76 distinct values in training, and the model overfitted them. Removing them improved average WAPE over three windows (0.141 → 0.138).
- Catalog attributes (`item_role`, categories, own brand), a per-product weekday profile and a last-year seasonal ratio: each made validation WAPE slightly worse, so the final set stays small and easy to explain.

**Not used:** traffic and stock (realised, unknown in the future; traffic mirrors sales), the promotion plan (no variation to learn from), `avg_historic_sales` and current prices (possible leakage).

## Part 3: Predictive Modeling

Compare a simple baseline with one proposed approach on a full historical 14-day forecast window. Explain your multi-day forecasting strategy and how you avoid using information from inside the validation window.

### Model and validation

Summary at the end of this section.


### 3.1 Validation design
We pretend to stand on **Sunday 23 August** and forecast the next 14 days, which we already know.

| Set | Origins | Target days |
|---|---|---|
| Train | Every Sunday, 2 Mar 2025 – 9 Aug 2026 (76 origins) | 3 Mar 2025 – 23 Aug 2026 |
| Validation | 23 Aug 2026 | **24 Aug – 6 Sep 2026** |

- The last training origin (9 Aug) has its last target on 23 Aug, so **no training target falls inside the validation window**, and validation features use data up to 23 Aug only.
- Each origin gives 3,920 products × 14 days. All 14 days are predicted at once (direct strategy): `horizon` tells the model how far ahead each row is, so no prediction is fed back as an input.
- Training starts in March 2025 so every rolling window already has enough history (data start 16 Sep 2024).

### 3.2 Baseline and metrics
**Baseline (seasonal naive):** each product sells what it sold on the **same weekday of the last week known at the origin**. For the 23 Aug origin, every Monday in the window gets Monday 17 Aug, every Tuesday gets Tuesday 18 Aug, and so on.

In [10]:
def baseline(f, origin):
    t, h = dates.get_loc(pd.Timestamp(origin)), f.horizon.to_numpy()
    last_same_weekday = t + h - 7 * np.ceil(h / 7).astype(int)
    return X[products.get_indexer(f.product_id), last_same_weekday]

def metrics(y, p):
    total = y.sum()                       # WAPE and bias are undefined when actual sales sum to 0
    return pd.Series({
        "WAPE": np.abs(y - p).sum() / total if total > 0 else np.nan,
        "MAE":  np.abs(y - p).mean(),
        "RMSE": np.sqrt(((y - p) ** 2).mean()),
        "Bias": (p.sum() - total) / total if total > 0 else np.nan,
    })

WAPE and bias are reported as **fractions** (0.10 = 10%). Bias > 0 means over-forecasting. If actual sales in a group sum to zero, WAPE and bias are reported as missing rather than infinite (this does not happen in our windows).

### 3.3 LightGBM
Training rows whose target day is a stock-out (`y` missing) are removed. We use the **L1 (absolute error) objective**, because WAPE and MAE are absolute-error metrics, and moderate settings without tuning.

In [11]:
import lightgbm as lgb

def train(last_origin):
    origins = pd.date_range("2025-03-02", last_origin, freq="7D")
    data = pd.concat([build_features(o) for o in origins])
    print(f"{len(data):,} rows built, {data.y.isna().sum():,} removed (stock-out targets)")
    data = data.dropna(subset=["y"])
    return lgb.LGBMRegressor(objective="l1", n_estimators=300, learning_rate=0.05, num_leaves=31,
                             min_child_samples=200, random_state=42, verbose=-1).fit(data[FEATURES], data.y)

model = train("2026-08-09")
val = build_features("2026-08-23")
val["baseline"] = baseline(val, "2026-08-23")
val["lightgbm"] = model.predict(val[FEATURES])

4,170,880 rows built, 288,360 removed (stock-out targets)


### 3.4 Results on the validation window (24 Aug – 6 Sep)

In [12]:
cohort = sql("""SELECT product_id FROM product_attributes
                 WHERE item_role = 'Focus article' AND product_category_2 = 'Tops'""").product_id
groups = {"All products (3,920)": val, f"Cohort: Focus article / Tops ({len(cohort)})": val[val.product_id.isin(cohort)]}

results = pd.concat({(g, m): metrics(d.y, d[m]) for g, d in groups.items() for m in ["baseline", "lightgbm"]}, axis=1).T.round(3)
results

WAPE    MAE   RMSE   Bias
All products (3,920)              baseline  0.218  0.864  1.482  0.099
                                  lightgbm  0.201  0.796  1.326  0.089
Cohort: Focus article / Tops (37) baseline  0.172  0.367  0.663  0.102
                                  lightgbm  0.168  0.358  0.594  0.101

### 3.5 Robustness: two earlier windows
One window can be lucky. The same comparison on the two previous 14-day windows (each with its own training cut):

In [13]:
rows = []
for origin, last_train in [("2026-07-26", "2026-07-12"), ("2026-08-09", "2026-07-26")]:
    m, f = train(last_train), build_features(origin)
    for name, pred in [("baseline", baseline(f, origin)), ("lightgbm", m.predict(f[FEATURES]))]:
        rows.append({"window_origin": origin, "model": name, **metrics(f.y, pred)})
pd.DataFrame(rows).round(3)

3,951,360 rows built, 288,360 removed (stock-out targets)


4,061,120 rows built, 288,360 removed (stock-out targets)


,window_origin,model,WAPE,MAE,RMSE,Bias
0,2026-07-26,baseline,0.113,0.508,0.919,-0.059
1,2026-07-26,lightgbm,0.101,0.457,0.776,-0.054
2,2026-08-09,baseline,0.100,0.437,0.857,0.039
3,2026-08-09,lightgbm,0.111,0.484,0.763,0.022


### Model and validation: summary
- **LightGBM beats the baseline on the validation window**, overall and for the cohort, on every metric (see 3.4). Across the three windows it wins two and loses one, with a lower average WAPE (≈0.138 vs 0.144), so the gain is real but modest.
- **Both models over-forecast the validation window by roughly 9–10%.** Shop-wide sales dropped from 1 September, after the validation origin. No feature known on 23 August could anticipate it. This matters for Part 4.
- **Choice:** LightGBM, because it is consistently at least as good as the baseline, the improvement holds beyond one window, and it uses only seven transparent features. The baseline remains a credible fallback.
- **Final forecast (Part 5):** the same model retrained with origins up to 23 August (targets up to 6 September) and applied at origin 6 September.

## Part 4: Forecast Evaluation & Merchandise Planner Investigation

Report WAPE, MAE, RMSE, and forecast bias for both approaches, overall and for the investigation cohort. State your metric conventions and zero-denominator handling.

Investigate the Focus article / Tops cohort question in the README. Include executable DuckDB SQL and its output, a compact table or chart, at least two plausible explanations, and a small sensitivity check. Separate observations from interpretations and assumptions.

### 4.1 Quantitative evaluation
Same validation window (24 Aug – 6 Sep), same metrics function as Part 3. WAPE and bias are **fractions**; bias > 0 means over-forecasting; when actual sales sum to zero, WAPE and bias are reported as missing (never happens here).

In [14]:
results

WAPE    MAE   RMSE   Bias
All products (3,920)              baseline  0.218  0.864  1.482  0.099
                                  lightgbm  0.201  0.796  1.326  0.089
Cohort: Focus article / Tops (37) baseline  0.172  0.367  0.663  0.102
                                  lightgbm  0.168  0.358  0.594  0.101

LightGBM is better than the baseline overall and for the cohort on every metric, but both **over-forecast by about 9–10%**, overall and for the cohort. Sales across the shop fell in the second half of the window (1–6 September), after the validation origin. That is exactly the period the planner is asking about.

### 4.2 Verify the planner's observation
Cohort: `item_role = 'Focus article'` and `product_category_2 = 'Tops'`. Periods: P1 = 10–23 Aug, P2 = 24 Aug – 6 Sep (same length, same weekdays). The clean version uses the `daily` table from Part 1; the raw version uses `transactions` with no cleaning at all.

In [15]:
sql("""
WITH cohort AS (SELECT product_id FROM product_attributes
                WHERE item_role = 'Focus article' AND product_category_2 = 'Tops'),
both_versions AS (
    SELECT 'clean' AS version, date, units FROM daily JOIN cohort USING (product_id)
    UNION ALL
    SELECT 'raw', time::DATE, sales_before_returns FROM transactions JOIN cohort USING (product_id)
)
SELECT version,
       (SELECT COUNT(*) FROM cohort)                                  AS products,
       SUM(units) FILTER (date BETWEEN '2026-08-10' AND '2026-08-23') AS p1_units,
       SUM(units) FILTER (date BETWEEN '2026-08-24' AND '2026-09-06') AS p2_units,
       p2_units - p1_units                                            AS change,
       ROUND(p2_units / p1_units - 1, 3)                              AS change_pct
FROM both_versions GROUP BY 1 ORDER BY 1
""")

,version,products,p1_units,p2_units,change,change_pct
0,clean,37,1179.0,1102.0,-77.0,-0.065
1,raw,37,1181.0,1104.0,-77.0,-0.065


**Observed:** the drop is real. The 37 products sold **1,179 units in P1 and 1,102 in P2 (−77 units, −6.5%)**. The raw data gives the same picture, so the result is not sensitive to cleaning. Coverage note: all 37 products belong to the 801 products with the Nov-25 – May-26 stock-out (Part 1.3); their stock was positive again from 24 May, so both periods are fully in stock.

### 4.3 Possible explanations
**A. A shop-wide or seasonal movement, not a Tops problem.** Same periods for the rest of the shop, and the same weeks one year earlier (aligned by weekday):

In [16]:
sql("""
WITH cohort AS (SELECT product_id FROM product_attributes
                WHERE item_role = 'Focus article' AND product_category_2 = 'Tops')
SELECT CASE WHEN product_id IN (SELECT product_id FROM cohort) THEN 'Cohort (37)' ELSE 'Rest of shop (3,883)' END AS grp,
       ROUND(SUM(units) FILTER (date BETWEEN '2026-08-24' AND '2026-09-06')
           / SUM(units) FILTER (date BETWEEN '2026-08-10' AND '2026-08-23') - 1, 3) AS change_2026,
       ROUND(SUM(units) FILTER (date BETWEEN '2025-08-25' AND '2025-09-07')
           / SUM(units) FILTER (date BETWEEN '2025-08-11' AND '2025-08-24') - 1, 3) AS change_same_weeks_2025
FROM daily GROUP BY 1 ORDER BY 1
""")

,grp,change_2026,change_same_weeks_2025
0,Cohort (37),-0.065,-0.021
1,"Rest of shop (3,883)",-0.096,-0.035


**B. Fewer strong voucher days.** Almost every order carries a voucher, and its intensity changes day by day for the whole shop (Part 1.5). Daily voucher share of order value, and cohort sales on strong-voucher days (≥ 7%) versus other days:

In [17]:
sql("""
WITH cohort AS (SELECT product_id FROM product_attributes
                WHERE item_role = 'Focus article' AND product_category_2 = 'Tops'),
voucher AS (SELECT time::DATE AS date, SUM(voucher_spending) / SUM(revenue + voucher_spending) AS voucher_share
            FROM transactions
            WHERE time BETWEEN '2026-08-10' AND '2026-09-06 23:59:59'
              AND sales_before_returns BETWEEN 1 AND 1000 AND revenue IS NOT NULL
            GROUP BY 1),
units AS (SELECT date, SUM(units) AS units FROM daily JOIN cohort USING (product_id) GROUP BY 1)
SELECT CASE WHEN date <= '2026-08-23' THEN 'P1' ELSE 'P2' END       AS period,
       ROUND(AVG(voucher_share), 3)                                 AS avg_voucher_share,
       SUM((voucher_share >= 0.07)::INT)                            AS strong_voucher_days,
       ROUND(AVG(units) FILTER (voucher_share >= 0.07), 1)          AS cohort_units_strong_days,
       ROUND(AVG(units) FILTER (voucher_share <  0.07), 1)          AS cohort_units_other_days
FROM voucher JOIN units USING (date)
GROUP BY 1 ORDER BY 1
""")

,period,avg_voucher_share,strong_voucher_days,cohort_units_strong_days,cohort_units_other_days
0,P1,0.055,6.0,96.5,75.0
1,P2,0.048,3.0,90.7,75.5


**C. Availability or data problems.** First, cohort products with at least one sale per day. Second, a suspicious weekend: shop-wide traffic for 29 and 30 August is identical, and we check whether sales repeat as well.

In [18]:
sql("""
WITH cohort AS (SELECT product_id FROM product_attributes
                WHERE item_role = 'Focus article' AND product_category_2 = 'Tops'),
selling AS (SELECT date, SUM((units > 0)::INT) AS products_selling
            FROM daily JOIN cohort USING (product_id)
            WHERE date BETWEEN '2026-08-10' AND '2026-09-06' GROUP BY 1)
SELECT CASE WHEN date <= '2026-08-23' THEN 'P1' ELSE 'P2' END AS period,
       MIN(products_selling) AS min_products_selling_per_day,
       ROUND(AVG(products_selling), 1) AS avg_products_selling_per_day
FROM selling GROUP BY 1 ORDER BY 1
""")

,period,min_products_selling_per_day,avg_products_selling_per_day
0,P1,36.0,36.7
1,P2,35.0,36.6


In [19]:
sql("""
SELECT active_since::DATE AS day, SUM(total_clicks) AS clicks, ROUND(SUM(marketing_cost), 2) AS marketing_cost
FROM traffic
WHERE active_since BETWEEN '2026-08-27' AND '2026-08-31'
GROUP BY 1 ORDER BY 1
""")

,day,clicks,marketing_cost
0,2026-08-27,3877722.0,157664.62
1,2026-08-28,4553049.0,187375.72
2,2026-08-29,4644886.0,190390.65
3,2026-08-30,4644886.0,190390.65
4,2026-08-31,5211270.0,212665.85


In [20]:
sql("""
WITH d AS (SELECT product_id, date, units FROM daily)
SELECT 'Sat 22 vs Sun 23 Aug' AS pair, ROUND(AVG((a.units = b.units)::INT), 3) AS share_products_identical
FROM d a JOIN d b USING (product_id) WHERE a.date = '2026-08-22' AND b.date = '2026-08-23'
UNION ALL
SELECT 'Sat 29 vs Sun 30 Aug', ROUND(AVG((a.units = b.units)::INT), 3)
FROM d a JOIN d b USING (product_id) WHERE a.date = '2026-08-29' AND b.date = '2026-08-30'
""")

,pair,share_products_identical
0,Sat 22 vs Sun 23 Aug,0.820
1,Sat 29 vs Sun 30 Aug,0.988


**Evidence summary**

| Explanation | Supporting evidence | Contradicting or missing evidence |
|---|---|---|
| **A. Shop-wide / seasonal** | Rest of the shop fell more (−9.6% vs −6.5%). The same weeks of 2025 also dipped (cohort −2.1%, rest −3.5%). | Last year's dip was smaller, so seasonality explains only part of it. |
| **B. Fewer strong voucher days** | P2 had 3 strong-voucher days vs 6 in P1 and a lower average voucher share; the cohort sells more on strong-voucher days. | The link is not one-to-one (some strong days show no lift). The planned 10% campaign from 1 Sep did not raise the realised voucher share, and the daily voucher plan for 7–20 Sep is unknown. |
| **C. Availability** | — | Contradicted: all 37 products sold almost every day in both periods; stock snapshots of 23 Jun and 1 Sep are positive. Missing: no stock snapshot inside August. |
| **C. Data quality (30 Aug)** | Shop-wide clicks and marketing cost are identical to the cent on 29 and 30 Aug, and 99% of products have identical units (vs 82% for the previous weekend): 30 Aug is probably a copied day. | Order ids differ between the two days. Its impact on the cohort is at most a few dozen units, so it cannot explain the whole drop. |

**Interpretation:** the drop is real but small, shared with the whole shop, and coincides with the seasonal pattern and with fewer strong voucher days. Nothing points to a problem specific to Tops (supply, data coverage). The data cannot fully separate seasonality from voucher intensity, so 4.4 tests directly whether drops like this tend to last.

### 4.4 Implication for the forecast and sensitivity check
**The assumption to test:** *when a group's sales fall over two weeks, the next two weeks stay at the lower level.* Cutting the forecast relies on it; our model implicitly assumes the opposite, because its 4-week averages pull the forecast back toward the earlier level.

**Historical test (no model involved).** For every Sunday since October 2024 and every group of products, take three consecutive 14-day blocks of actual sales: *before*, *last* and *next*. Where *last* < *before* (a drop), compare three simple guesses for *next*:
- **returns** to the earlier level (`before`), what not cutting assumes;
- **halfway** between the two;
- **stays** at the recent level (`last`), what the planner's cut assumes.

Groups: the cohort, each of the 74 `product_category_2` categories, and the whole shop. Windows touching the Nov–May stock-out are excluded. Error is WAPE over all cases (sum of absolute errors / sum of actual *next* sales).

In [21]:
sql("""
WITH groups AS (
    SELECT 'Cohort (Focus article / Tops)' AS scope, product_category_2 AS grp, product_id
    FROM product_attributes WHERE item_role = 'Focus article' AND product_category_2 = 'Tops'
    UNION ALL SELECT 'All 74 categories', product_category_2, product_id FROM product_attributes
    UNION ALL SELECT 'Whole shop', 'all', product_id FROM product_attributes
),
group_daily AS (
    SELECT scope, grp, date, SUM(units) AS units, BOOL_OR(stockout) AS stockout
    FROM daily JOIN groups USING (product_id) GROUP BY ALL
),
origins AS (SELECT range::DATE AS origin FROM range(DATE '2024-10-13', DATE '2026-08-24', INTERVAL 7 DAY)),
blocks AS (
    SELECT scope, grp, origin,
           SUM(units) FILTER (date BETWEEN origin - 27 AND origin - 14) AS before_14,
           SUM(units) FILTER (date BETWEEN origin - 13 AND origin)      AS last_14,
           SUM(units) FILTER (date BETWEEN origin + 1  AND origin + 14) AS next_14,
           BOOL_OR(stockout) AS any_stockout
    FROM origins JOIN group_daily ON date BETWEEN origin - 27 AND origin + 14
    GROUP BY ALL
),
drops AS (
    SELECT *, ABS(before_14 - next_14)                 AS err_returns,
              ABS((before_14 + last_14) / 2 - next_14) AS err_halfway,
              ABS(last_14 - next_14)                   AS err_stays
    FROM blocks WHERE NOT any_stockout AND last_14 < before_14
)
SELECT scope, COUNT(*) AS cases,
       ROUND(SUM(err_returns) / SUM(next_14), 3)                       AS wape_returns,
       ROUND(SUM(err_halfway) / SUM(next_14), 3)                       AS wape_halfway,
       ROUND(SUM(err_stays)   / SUM(next_14), 3)                       AS wape_stays,
       ROUND(AVG((err_stays < LEAST(err_returns, err_halfway))::INT), 3) AS share_stays_closest
FROM drops GROUP BY 1 ORDER BY cases
""")

,scope,cases,wape_returns,wape_halfway,wape_stays,share_stays_closest
0,Whole shop,26,0.029,0.023,0.026,0.500
1,Cohort (Focus article / Tops),40,0.091,0.069,0.051,0.825
2,All 74 categories,3273,0.104,0.079,0.060,0.741


**Forecast for the cohort, 7–20 September** (model retrained with origins up to 23 Aug and applied at the 6 Sep origin; exported in Part 5):

In [22]:
final_model = train("2026-08-23")
forecast = build_features("2026-09-06")
forecast["predicted_sales"] = final_model.predict(forecast[FEATURES]).clip(min=0)

cohort_daily = daily[daily.product_id.isin(cohort)]
p1 = cohort_daily.loc[cohort_daily.date.between("2026-08-10", "2026-08-23"), "units"].sum()
p2 = cohort_daily.loc[cohort_daily.date.between("2026-08-24", "2026-09-06"), "units"].sum()
pd.Series({
    "P1 actual (10-23 Aug)": p1,
    "P2 actual (24 Aug-6 Sep)": p2,
    "Model forecast (7-20 Sep)": forecast.loc[forecast.product_id.isin(cohort), "predicted_sales"].sum(),
    "Recommended planning level (recent level = P2)": p2,
}).round(0).rename("cohort units")

4,280,640 rows built, 288,360 removed (stock-out targets)


P1 actual (10-23 Aug)                             1179.0
P2 actual (24 Aug-6 Sep)                          1102.0
Model forecast (7-20 Sep)                         1176.0
Recommended planning level (recent level = P2)    1102.0
Name: cohort units, dtype: float64

**Implication**
- **Accuracy evidence:** after a two-week drop, sales usually **stay** at the lower level. *Stays* is the closest guess in most cases, for the cohort and across the 74 categories, and has the lowest error (table above). The whole shop is the exception: there, drops are as likely to reverse.
- **Why the model is too high here:** it treats group-level drops as temporary. This matches the validation window, where it over-forecast the cohort by about 10% (4.1).
- **Sensitivity:** moving the cohort from the model forecast (~1,176) to the recent level (~1,102) changes the plan by about 75 units (−6%).
- **Decision:** for planning the cohort total, use the recent level, **about 1,100 units for 7–20 September**. `predictions.csv` keeps the model's product-level forecasts: a uniform cut makes individual products less accurate, because not every product in the group falls by the same amount.
- **Limits:** the test uses simple 14-day blocks, not the model, and consecutive Sundays share weeks, so the cases are not independent. The pattern is consistent across groups and drop sizes, which is what makes it convincing.

### Client memo (maximum 300 words, 3–5 bullets)

- **Recommendation: plan Focus article / Tops at the recent level, about 1,100 units for 7–20 September (−6% versus our model's 1,176), and review after the first week.**
- **Observed:** the drop is real. The 37 products sold 1,179 units in 10–23 Aug and 1,102 in 24 Aug–6 Sep (−6.5%, the same with raw data). It is not a supply problem: every product kept selling almost daily and stock covers months.
- **Observed:** such drops usually last. Since October 2024, after a two-week drop the following two weeks stayed closer to the lower level in most cases, for this cohort (40 cases) and across all 74 product categories (over 3,000 cases). *Interpretation:* this looks like a seasonal decline shared by the whole shop (other products fell 9.6%), not a Tops-specific issue.
- **Uncertainty:** part of the drop coincides with fewer strong-voucher days (3 vs 6), a driver that could reverse; 30 Aug also looks like a copied day in the data. The information that would most change the decision: the daily voucher plan for 7–20 Sep and whether the 10% and 5% vouchers stack (11–16 Sep).
- **Commercial trade-off.** *Assumptions:* a lost sale costs about €9 of profit and an unsold unit at most about €2 (averages since June). If demand rebounds to the model level, the lower plan misses about 75 units (≈€670), but current stock (at least 412 units per product) still covers that demand, so the risk is mainly to targets rather than lost sales. If we kept the model plan and the drop lasts, about 75 units of excess (≈€150). **I would change the recommendation** if 7–13 Sep sales run 5% or more above the lower plan, or if strong voucher days are scheduled; then return to the model forecast for the second week.

## Part 5: Generate Submission Predictions

Generate predictions for September 7–20, 2026, using the exact product/date pairs in `sample_prediction.csv`. Save the required `predictions.csv` file.

### Export and checks

Run the notebook top to bottom (about 10 minutes). The cell below writes `predictions.csv` and checks row count, missing values and negatives.


Uses `forecast` from 4.4 (final model trained with origins up to 23 Aug, applied at the 6 Sep origin). The output is joined to `sample_prediction.csv` so it covers exactly the required pairs.

In [23]:
sample = pd.read_csv("sample_prediction.csv", dtype={"product_id": str}, parse_dates=["date"])
predictions = (sample[["product_id", "date"]]
               .merge(forecast[["product_id", "date", "predicted_sales"]], on=["product_id", "date"], how="left"))

assert len(predictions) == len(sample) == 3920 * 14, "row count"
assert predictions.predicted_sales.notna().all(), "missing predictions"
assert (predictions.predicted_sales >= 0).all(), "negative predictions"

predictions["date"] = predictions.date.dt.strftime("%Y-%m-%d")
predictions.round(4).to_csv("predictions.csv", index=False)
print(f"{len(predictions):,} rows saved | {predictions.date.min()} to {predictions.date.max()} | "
      f"total {predictions.predicted_sales.sum():,.0f} units")
predictions.head()

54,880 rows saved | 2026-09-07 to 2026-09-20 | total 208,664 units


,product_id,date,predicted_sales
0,2_x_669800,2026-09-07,4.223256
1,2_x_669800,2026-09-08,3.667929
2,2_x_669800,2026-09-09,3.733360
3,2_x_669800,2026-09-10,3.753391
4,2_x_669800,2026-09-11,3.734717


### Unfinished work and next steps
- **Group-level trend in the model.** The model treats drops at category level as temporary
  and over forecast the cohort by ~10% in validation. Adding features such as the recent
  change of the product's category (last 14 days vs the previous 14) would let the model
  learn the pattern found in 4.4, instead of applying it as a manual adjustment.
- **Voucher intensity as a driver.** Daily voucher share explains part of the sales
  variation (4.3 B). With the daily voucher plan for the forecast period, it could become a
  known in advance feature.
- **Data confirmation.** Confirm with the data team whether 30 August was copied or imputed,
  and whether the 10% and 5% vouchers stack on 11–16 September.
- **Light hyperparameter tuning with rolling validation.** Tune a few LightGBM settings
  (number of trees, leaves, learning rate) across several 14 day windows, not a single one,
  so any gain is not specific to one period.
- **Product-level check of the group adjustment.** Measure in the notebook how much a uniform
  cut changes product level accuracy for the cohort, to decide whether the planning
  adjustment should also be applied to `predictions.csv`.

## Evaluation Criteria

Submissions are assessed on data reasoning (30%), forecasting and validation (25%), business judgment (25%), and communication (20%). Runnable code and the required prediction file are basic deliverables. Model complexity and extensive tuning do not earn additional credit by themselves.
